# US RIA-like synthetic household data

This notebook walks through schema, priors, generation, validation, scenario coverage, anomaly detection and reporting.


## Official anchors

- Census reports median household income of **$80,610** in 2023.
- Census SIPP reports median household wealth of **$191,100** in 2023 and 90th percentile wealth of **$1,806,000**.
- BLS reports average income before taxes of **$104,207** and average expenditures of **$78,535** for 2024.
- SCF 2022 is the latest full public Federal Reserve SCF release.

These are used as anchors only. The target segment is shifted upward to represent affluent, advisor-served households.


In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
CFG = ROOT / 'config'
SRC = ROOT / 'src'
ART = ROOT / 'artifacts'
GEN = ART / 'generated'


## 1. Schema


In [ ]:
schema = json.loads((CFG / 'schema.json').read_text())
schema_df = pd.DataFrame([
    {'entity': entity, 'field': f['name'], 'type': f['type']}
    for entity, spec in schema['entities'].items()
    for f in spec['fields']
])
schema_df.head(25)


## 2. Priors and assumptions


In [ ]:
assumptions = json.loads((CFG / 'priors_assumptions.json').read_text())
assumptions['official_anchors']


In [ ]:
subprocess.run([sys.executable, str(SRC / '02_compute_priors.py')], check=True)
priors = json.loads((ART / 'computed_priors.json').read_text())
priors['continuous_targets']


## 3. Generate data

Generation is scenario-based and uses conditional logic. Dates are derived from lifecycle variables rather than sampled independently.


In [ ]:
subprocess.run([sys.executable, str(SRC / '03_generate_data.py'), '--n-households', '5000'], check=True)
hh = pd.read_csv(GEN / 'households.csv')
people = pd.read_csv(GEN / 'people.csv')
assets = pd.read_csv(GEN / 'assets.csv')
liab = pd.read_csv(GEN / 'liabilities.csv')
hh.head()


## 4. Distribution checks

Categorical stability is measured with JS divergence.
Continuous stability is measured with **Population Stability Index (PSI)**, which is common in financial-services monitoring.

PSI interpretation:
- < 0.10: no meaningful shift
- 0.10 to 0.25: moderate shift
- >= 0.25: large shift


In [ ]:
subprocess.run([sys.executable, str(SRC / '04_validate_and_score.py')], check=True)
metrics = pd.read_csv(GEN / 'distance_to_priors.csv')
metrics


In [ ]:
fig = plt.figure(figsize=(8,5))
plt.hist(hh['annual_household_gross_income'], bins=40)
plt.xscale('log')
plt.title('Household annual gross income')
plt.show()

fig = plt.figure(figsize=(8,5))
plt.hist(hh['investable_assets_total'], bins=40)
plt.xscale('log')
plt.title('Investable assets total')
plt.show()

fig = plt.figure(figsize=(8,5))
plt.hist(hh['net_worth_proxy'], bins=40)
plt.xscale('log')
plt.title('Net worth proxy')
plt.show()


## 5. Business-rule checks


In [ ]:
rules = pd.read_csv(GEN / 'rule_violations.csv')
rules['rule_violation'].value_counts().head(10)


## 6. Scenario coverage
The generator explicitly covers lifecycle and advisory-relevant archetypes.


In [ ]:
scenario = pd.read_csv(GEN / 'scenario_coverage.csv')
scenario


In [ ]:
fig = plt.figure(figsize=(10,5))
scenario.sort_values('count', ascending=False).plot(x='scenario', y='count', kind='bar', legend=False)
plt.title('Scenario coverage')
plt.show()


## 7. Anomaly detection

After generation and metric calculation, we train a **PyTorch autoencoder** on household-level numeric features.
We then inspect the 5 most anomalous households by reconstruction error.


In [ ]:
subprocess.run([sys.executable, str(SRC / '05_autoencoder_anomalies.py')], check=True)
top5 = pd.read_csv(GEN / 'top5_anomalous_households.csv')
top5[['household_id','scenario','annual_household_gross_income','investable_assets_total','loan_outstanding_total','reconstruction_error']]


These households should be inspected manually. Typical reasons for anomaly scores include rare combinations such as:
- unusually high debt for the scenario,
- very high assets with relatively low income,
- unusually large alimony burden,
- rare lifecycle / financial-state combinations.


## 8. Final report artifacts


In [ ]:
subprocess.run([sys.executable, str(SRC / '06_report.py')], check=True)
